# Gradio × GPT API 단계별 실습

이 노트북은 **Gradio를 처음 보는 사람**도 따라올 수 있도록
아주 작은 것부터 시작해서 한 단계씩 살을 붙여 나갑니다.

목표는 두 가지예요.

1. **Gradio가 뭔지** 감을 잡는다 (화면 만드는 도구)
2. **GPT API의 옵션값**(`temperature`, `max_tokens`)이
   결과를 어떻게 바꾸는지 직접 손으로 돌려보며 확인한다

> 마지막에는 `감기`, `허리아픔` 같은 **단어 하나만 바꿔서**
> 그 증상에 좋은 관리법을 받아보는 작은 앱을 만듭니다.

진행 순서  (Step1,~4 까지는 앞서 질문한 내용을 기억하지 못함.)

- **Step1** — GPT API 호출, 가장 기본
- **Step2** — Gradio 초간단 버전 (입력칸 1개, 출력칸 1개)
- **Step3** — `temperature`, `max_tokens` 슬라이더로 옵션 조절
- **Step4** — 증상 단어만 바꿔서 관리법 받기
- **Step5** — 히스토리 기억하기,    예: 감기증상은   엔터 ,   예방법은  입력하면 감기에 대한 예방법 출력


## [0단계] — 준비물 설치

먼저 필요한 라이브러리 두 개를 설치합니다.

- `openai` : GPT API를 부르는 공식 라이브러리
- `gradio` : 코드 몇 줄로 웹 화면을 만들어주는 도구
- `python-dotenv` : `.env` 파일에서 API 키를 읽어오는 도구

`!`로 시작하는 줄은 파이썬이 아니라 **터미널 명령어**라는 뜻이에요.
`-q`는 설치 로그를 조용히(quiet) 줄이라는 옵션입니다.


In [ ]:
!pip install -q openai gradio python-dotenv

In [ ]:
!pip install -q --upgrade gradio

### `.env`에서 키 불러오기
.env 파일을 사용할수 있게 준비합니다.

`load_dotenv()`가 `.env` 파일을 읽어서
키를 프로그램의 **환경변수**에 올려줍니다.
그 다음 `os.getenv(...)`로 그 값을 꺼내 씁니다.

키가 잘 들어왔는지 **앞 7글자만** 출력해서 확인해볼게요.
(전체를 출력하면 키가 노출되니 일부만 봅니다.)


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                              # .env 읽기
api_key = os.getenv("OPENAI_API_KEY")      # 키 꺼내기

# 키가 제대로 들어왔는지 안전하게 확인
if api_key and api_key.startswith("sk-"):
    print("키 로드 성공:", api_key[:7] + "...")
else:
    print("키를 못 읽었습니다. .env 파일과 키 값을 다시 확인하세요.")

# OpenAI 클라이언트 만들기 (이 client로 앞으로 API를 부릅니다)
client = OpenAI(api_key=api_key)

키 로드 성공: sk-proj...


## [STep1] — GPT API 호출, 가장 기본

Gradio를 붙이기 전에, **API 자체가 잘 도는지** 먼저 확인합니다.
화면 없이 파이썬 함수로만 GPT를 한 번 불러볼게요.

호출 구조는 늘 똑같습니다.

```
client.chat.completions.create(
    model = 어떤 모델을 쓸지,
    messages = 대화 내용,
)
```

`messages`는 리스트이고, 각 항목은 `role`과 `content`를 가집니다.
지금은 사용자(`user`) 한 명이 질문 하나 던지는 가장 단순한 형태예요.


In [ ]:
def ask_gpt(question):
    res = client.chat.completions.create(
        model="gpt-4o-mini",                       # 가볍고 빠른 모델
        messages=[
            {"role": "user", "content": question}, # 사용자 질문
        ],
    )
    # 응답 객체에서 실제 답변 텍스트만 꺼냅니다
    return res.choices[0].message.content


# 바로 테스트
print(ask_gpt("한 문장으로 자기소개 해줘."))

안녕하세요, 저는 다양한 주제에 대해 정보와 도움을 제공하는 인공지능 언어 모델입니다!


여기까지 답이 잘 나왔다면 **API 연결은 성공**입니다.
이제 이 함수에 화면(Gradio)을 입혀봅시다.


## [Step2] — Gradio 초간단 버전

Gradio의 핵심 아이디어는 딱 하나입니다.

> **"파이썬 함수 하나"를 주면, 그걸 웹 화면으로 만들어준다.**

가장 많이 쓰는 게 `gr.Interface`인데, 세 가지만 알려주면 됩니다.

- `fn` : 실행할 함수 (입력을 받아 출력을 돌려주는 함수)
- `inputs` : 입력칸의 종류 (`"text"` = 글자 입력칸)
- `outputs` : 출력칸의 종류 (`"text"` = 글자 출력칸)

그리고 `.launch()`를 부르면 화면이 뜹니다.

> **Colab에서는 `share=True`가 중요해요.**
> 이걸 넣어야 `...gradio.live` 라는 **공개 링크**가 생겨서
> 화면이 제대로 보입니다. 안 넣으면 화면이 안 뜨기도 해요.


In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=ask_gpt,
    inputs="text",
    outputs="text",
    title="가장 간단한 GPT 데모",
)

demo.launch(
    share=True)

print("여기까지 실행됨")


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://694c12e085569355c3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


여기까지 실행됨


In [ ]:
# 위의 실행된 그라디오 종료
# 이 셀 실행하기 전까지는 위의 그라디오 메세지창에 입력하고 결과보고 계속 할수 있음
demo.close() # 종료

Closing server running on port: 7860


실행하면 입력칸에 아무 질문이나 적고 **Submit**을 누르면
GPT 답이 출력칸에 뜹니다.

여기까지가 "Gradio 초간단" 버전이에요.
**함수 하나 → Interface로 감싸기 → launch**, 이 흐름이 전부입니다.

> 다 쓴 화면은 셀 출력 위쪽의 정지 버튼이나
> `demo.close()`로 닫을 수 있어요.


## [Step3] — 옵션값 조절하기 (`temperature`, `max_tokens`)

이제 이 노트북의 **진짜 목적**입니다.
GPT API의 두 옵션을 화면에서 직접 바꿔보며 효과를 확인해요.

- **`temperature`** (0 ~ 2)
  답변의 **무작위성/창의성**을 조절합니다.
  - `0`에 가까울수록 → 매번 비슷하고 안정적인 답
  - 높을수록 → 다양하고 자유분방한 답 (대신 산만해질 수 있음)

- **`max_tokens`**
  답변의 **최대 길이**입니다 (토큰 ≈ 단어 조각).
  작게 주면 짧게 끊기고, 크게 주면 길게 답할 수 있어요.

함수가 이제 **인자 3개**(질문, temperature, max_tokens)를 받습니다.
그리고 Gradio `inputs`에는 그 3개를 각각 입력 위젯으로 연결합니다.

- `gr.Textbox(...)` : 글자 입력칸
- `gr.Slider(...)` : 드래그로 숫자를 고르는 슬라이더

> **중요:** `inputs` 리스트의 순서가
> 함수의 인자 순서와 **그대로 짝이 맞아야** 합니다.
> (첫 번째 입력 → 첫 번째 인자, ...)


**이렇게 실험해보세요.**

1. 같은 질문을 `temperature=0`으로 두 번 → 거의 같은 답이 나옴
2. 같은 질문을 `temperature=1.5`로 두 번 → 매번 다른 답
3. `max_tokens=50`으로 → 답이 중간에 짧게 끊김
4. `max_tokens=800`으로 → 길고 자세한 답

옵션값이 결과를 어떻게 바꾸는지 **몸으로 느끼는 게** 이 단계의 핵심이에요.


In [ ]:
def ask_with_options(question, temperature, max_tokens):
    res = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": question},
        ],
        temperature=temperature,        # 슬라이더에서 받은 값
        max_tokens=int(max_tokens),     # 슬라이더 값은 실수라 int로 변환
    )
    return res.choices[0].message.content


demo = gr.Interface(
    fn=ask_with_options,
    inputs=[
        gr.Textbox(label="질문", value="우주에 대해 알려줘"),
        gr.Slider(0, 2, value=0.7, step=0.1, label="temperature (창의성)"),
        gr.Slider(50, 1000, value=300, step=50, label="max_tokens (길이)"),
    ],
    outputs=gr.Textbox(label="GPT 답변"),
    title="옵션 실험실: temperature & max_tokens",
    description="슬라이더를 바꿔가며 같은 질문이 어떻게 달라지는지 확인하세요.",
)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://7ac2e4aac08784c8d0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
demo.close() # 종료

Closing server running on port: 7860


## [Step4] — 증상 단어만 바꿔서 관리법 받기

마지막으로, **질문 변수 하나만 바꾸는** 작은 응용입니다.
`감기`, `허리아픔`처럼 증상 단어를 넣으면
그 증상에 좋은 관리법을 받아봅니다.

특별한 건 없어요. 사용자가 넣은 단어를
`f"..."` 문자열 안에 끼워 넣어서 질문 문장을 만들 뿐입니다.

```
f"{symptom} 증상에 좋은 관리법을 알려줘."
```

여기에 3단계의 옵션 슬라이더도 그대로 붙여서,
**증상 + temperature + max_tokens**를 함께 조절할 수 있게 했습니다.

> 입력 위젯에 `gr.Radio`를 쓰면 미리 정해둔 단어 중에서 **골라서**
> 넣을 수도 있어요. 직접 타이핑하고 싶으면 `gr.Textbox`를 쓰면 됩니다.
> 아래는 둘 다 쓸 수 있게 Radio로 예시 단어를 주되,
> 직접 입력도 가능하게 했습니다.


In [ ]:
def symptom_care(symptom, temperature, max_tokens):
    # 증상 단어를 질문 문장에 끼워 넣습니다 (여기가 "변수 하나만 바꾸는" 부분)
    question = f"{symptom} 증상에 좋은 생활 관리법을 알려줘."

    res = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": question},
        ],
        temperature=temperature,
        max_tokens=int(max_tokens),
    )
    return res.choices[0].message.content


demo = gr.Interface(
    fn=symptom_care,
    inputs=[
        gr.Textbox(label="증상 단어", value="감기",
                   placeholder="예: 감기, 허리아픔, 두통, 소화불량"),
        gr.Slider(0, 2, value=0.7, step=0.1, label="temperature"),
        gr.Slider(50, 1000, value=400, step=50, label="max_tokens"),
    ],
    outputs=gr.Textbox(label="관리법"),
    title="증상별 관리법 도우미",
    description="증상 단어 하나만 바꿔서 관리법을 받아보세요. (의학적 진단이 아닌 참고용입니다)",
)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://ad023fe97083721e61.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
demo.close() # 종료

Closing server running on port: 7861


## [Step5] — 대화처럼 주고받기 (히스토리 유지 + Clear)

지금까지는 질문을 넣으면 답이 **한 번** 나오고 끝이었어요.
이번엔 채팅처럼 **입력 → 출력 → 입력 → 출력**이 계속 쌓이고,
**Clear 버튼을 누르기 전까지** 이전 대화가 화면에 남아 있게 만듭니다.

이럴 때는 `gr.Interface` 대신 **`gr.ChatInterface`**를 씁니다.
이름 그대로 "채팅 화면"을 자동으로 만들어줘요.
- 대화 기록이 화면에 누적되고
- 입력칸, 전송 버튼, **Clear 버튼**까지 알아서 만들어 줍니다.

`ChatInterface`에 넘기는 함수는 인자를 **두 개** 받아야 합니다.

- def함수

- `message` : 사용자가 방금 입력한 새 메시지
- `history` : 지금까지 주고받은 대화 기록 (Gradio가 자동으로 챙겨줌)

우리는 이 `history`를 GPT가 알아듣는 `messages` 형식으로 바꿔서
**이전 대화 맥락까지 기억하는** 챗봇을 만들 거예요.
(맥락이 쌓이니 "방금 그거 더 자세히" 같은 말도 알아듣습니다.)

> `history`의 형식은 Gradio 버전에 따라 조금 다릅니다.
> 최신 버전은 `{"role": ..., "content": ...}` 딕셔너리 리스트라서
> 아래 코드는 그 형식(`type="messages"`)에 맞춰 작성했어요.

In [ ]:
def chat_fn(message, history):
    # 1) GPT에게 보낼 메시지 목록을 만듭니다.
    #    맨 앞에 시스템 지시(역할 설정)를 한 줄 넣어둡니다.
    messages = [
        {"role": "system", "content": "너는 친절한 건강 생활 도우미야. 참고용 정보를 쉽게 설명해줘."},
    ]
    # 2) 지금까지의 대화 기록(history)을 그대로 이어 붙입니다.
    #    → 이게 있어야 GPT가 이전 맥락을 기억합니다.
    for turn in history:
        messages.append({"role": turn["role"], "content": turn["content"]})
    # 3) 사용자가 방금 보낸 새 메시지를 마지막에 추가합니다.
    messages.append({"role": "user", "content": message})
    # 4) GPT 호출
    res = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.7,
    )
    return res.choices[0].message.content


# ChatInterface가 채팅 화면(누적 대화 + Clear 버튼)을 자동으로 만들어 줍니다.
demo = gr.ChatInterface(
    fn=chat_fn,
    title="건강 관리 챗봇",
    description="증상이나 궁금한 점을 계속 물어보세요. Clear를 누르면 대화가 초기화됩니다.",
)

demo.launch(share=True,height=800) # 높이조정

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://980b51ecabf8eda2c9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 

In [ ]:
demo.close() # 종료

Closing server running on port: 7862


### 정리

이 노트북에서 한 일

- `.env`로 API 키를 **안전하게** 불러왔다
- GPT API를 함수로 호출하는 **기본 구조**를 익혔다
- Gradio의 핵심 = **함수를 Interface로 감싸서 launch**
- `temperature`, `max_tokens`를 슬라이더로 바꿔보며 효과를 확인했다
- 증상 단어만 바꿔 넣는 **간단한 응용 앱**을 만들었다

> ⚠️ 결과는 GPT가 생성한 일반 정보이므로 참고용입니다.
> 실제 증상은 전문 의료인과 상담하세요.

수고하셨습니다! 슬라이더를 이리저리 바꿔보며
옵션값 감각을 익히는 걸 추천합니다.
